# 3 — The search-budget ablation

The paper's central methodological result, and the cheapest to check.

The eight-qubit advantage came from picking the best of 264 feature-map
configurations. What if the quantum side had only been allowed to try $B$ of
them?

For each budget $B$ we draw $B$ configurations at random, pick the best **on
development data**, and score that pick on held-out data. Repeating gives the
distribution of margins a researcher with budget $B$ would have reported.

In [ ]:
#@title Install and import
%pip install -q qiskit qiskit-aer scikit-learn matplotlib pandas scipy

import json, time, itertools
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import adjusted_rand_score, roc_auc_score
from sklearn.metrics.pairwise import rbf_kernel, laplacian_kernel

sim = AerSimulator()
RES = "../results"
print("imports ok")

In [ ]:
#@title The stored ablation
B = json.load(open(f"{RES}/budget.json"))
print(f"ablation performed at n = {B['n_qubits']} qubits")
print(f"classical kernel held-out mean ARI = {B['classical_mean']:.4f}\n")

rows = [{"budget": b,
         "mean margin": f"{B['results'][str(b)]['margin_mean']:+.4f}",
         "sd": f"{B['results'][str(b)]['margin_sd']:.4f}",
         "% draws quantum leads": f"{100*B['results'][str(b)]['frac_positive']:.0f}%"}
        for b in B["budgets"]]
display(pd.DataFrame(rows).set_index("budget"))

In [ ]:
#@title Plot it
bs = B["budgets"]
m = [B["results"][str(b)]["margin_mean"] for b in bs]
sd = [B["results"][str(b)]["margin_sd"] for b in bs]
fp = [100*B["results"][str(b)]["frac_positive"] for b in bs]

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
ax[0].axhline(0, color="k", lw=.8)
ax[0].errorbar(bs, m, yerr=sd, fmt="o-", capsize=3, color="#1F4E79")
ax[0].set_xscale("log"); ax[0].set_xlabel("quantum search budget")
ax[0].set_ylabel("quantum - classical (ARI)")
ax[1].plot(bs, fp, "o-", color="#E36C09"); ax[1].axhline(50, ls=":", color="k")
ax[1].set_xscale("log"); ax[1].set_xlabel("quantum search budget")
ax[1].set_ylabel("% of draws where quantum leads"); ax[1].set_ylim(0, 105)
for a in ax: a.grid(alpha=.25)
plt.tight_layout(); plt.show()

## What this shows

The margin is a **monotone function of how many circuits you tried**. At small
budgets the quantum kernel is behind on average; the sign flips around 50
configurations; the headline figure is only reached at the full 264.

Since the classical baseline was given 105 configurations, the honest reading
is that at comparable budgets the two are indistinguishable.

The control costs almost nothing to run and was, here, decisive. We would
encourage its routine use.